In [1]:
import sys
import asyncio
import nest_asyncio

# 1. Apply the Jupyter patch (allows nested loops)
nest_asyncio.apply()

# 2. Apply the Windows Fix (Required for Playwright)
# Without this, you get 'NotImplementedError' on Windows
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("✅ Setup complete. Windows Policy set and Asyncio patched.")

✅ Setup complete. Windows Policy set and Asyncio patched.


In [2]:
# Cell 2: Native Async Scraper (Full Description + Structured Data)
import asyncio
import json
import hashlib
import re
import random
import chromadb
from playwright.async_api import async_playwright
from playwright_stealth import stealth_async
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
CHROMA_PATH = "./joblens_db"
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

class JobLensScraper:
    def __init__(self):
        self.chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
        self.collection = self.chroma_client.get_or_create_collection(name="job_listings")
        print("Loading AI Model...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        
        existing_data = self.collection.get()
        self.existing_ids = set(existing_data['ids'])
        print(f"Loaded {len(self.existing_ids)} existing jobs.")

    def normalize_text(self, text):
        if not text: return ""
        return re.sub(r'[^a-z0-9]', '', text.lower())

    def generate_job_id(self, title, company):
        raw_sig = f"{self.normalize_text(title)}|{self.normalize_text(company)}"
        return hashlib.md5(raw_sig.encode()).hexdigest()

    def is_duplicate(self, job_id):
        return job_id in self.existing_ids

    def save_jobs(self, jobs):
        if not jobs: return
        new_ids, new_docs, new_metas, new_embeddings = [], [], [], []
        
        print(f"   Processing {len(jobs)} candidates for database...")
        for job in jobs:
            job_id = self.generate_job_id(job['title'], job['company'])
            if self.is_duplicate(job_id): continue 
            
            # Combine Title + Skills + Description for a rich embedding
            skills_str = ", ".join(job.get('skills', []))
            text_for_embedding = f"{job['title']} at {job['company']}. Skills: {skills_str}. {job.get('description', '')}"
            
            new_ids.append(job_id)
            new_docs.append(text_for_embedding)
            
            # --- METADATA STRATEGY ---
            # 1. We keep 'description' inside 'json_detailed' (full text)
            # 2. We add the NEW structured fields as separate columns
            meta = {
                "source": job['source'], 
                "title": job['title'], 
                "company": job['company'],
                "location": job['location'],
                "job_page_link": job['job_page_link'],
                "apply_link": job['apply_link'],
                "posted_time": job['posted_time'],
                
                # NEW STRUCTURED DATA (Added alongside everything else)
                "experience_level": job.get('experience_level', 'Not Specified'),
                "employment_type": job.get('employment_type', 'Not Specified'),
                "skills_list": ", ".join(job.get('skills', [])), # Stored as string
                
                # Snippets for quick preview (Full text is in json_detailed)
                "description_snippet": job.get('description', '')[:200],
                
                # Full Object (Contains the FULL Description + All new fields)
                "json_detailed": json.dumps(job) 
            }
            new_metas.append(meta)
            new_embeddings.append(self.model.encode(text_for_embedding).tolist())
            self.existing_ids.add(job_id)

        if new_ids:
            self.collection.upsert(ids=new_ids, embeddings=new_embeddings, metadatas=new_metas, documents=new_docs)
            print(f"   ✅ Success: Added {len(new_ids)} NEW jobs to DB.")
        else:
            print("   ⚠️ All jobs were duplicates.")

# --- HELPER: TEXT PARSER ---
def parse_description(full_text):
    """
    Extracts requirements/responsibilities from the full text 
    WITHOUT modifying the original description.
    """
    lower_text = full_text.lower()
    
    req_start = -1
    if "requirements" in lower_text: req_start = lower_text.find("requirements")
    elif "qualifications" in lower_text: req_start = lower_text.find("qualifications")
    
    resp_start = -1
    if "responsibilities" in lower_text: resp_start = lower_text.find("responsibilities")
    elif "duties" in lower_text: resp_start = lower_text.find("duties")
    
    requirements = ""
    responsibilities = ""
    
    if req_start != -1 and resp_start != -1:
        if req_start < resp_start:
            requirements = full_text[req_start:resp_start]
            responsibilities = full_text[resp_start:]
        else:
            responsibilities = full_text[resp_start:req_start]
            requirements = full_text[req_start:]
    elif req_start != -1:
        requirements = full_text[req_start:]
    elif resp_start != -1:
        responsibilities = full_text[resp_start:]
        
    return requirements.strip(), responsibilities.strip()

# --- HELPER: GET FULL DETAILS ---
async def get_job_details(context, link, source):
    """
    Returns a dictionary containing:
    1. 'description': The FULL raw text (Unchanged)
    2. 'apply_link': The direct apply link
    3. New Structured Keys: skills, experience_level, etc.
    """
    if not link: return {}
    
    await asyncio.sleep(random.uniform(1.0, 2.5))
    page = await context.new_page()
    
    # Initialize with defaults
    data = {
        "description": "",          # FULL TEXT
        "requirements": "",         # EXTRACTED
        "responsibilities": "",     # EXTRACTED
        "skills": [],               # EXTRACTED
        "experience_level": "Not Specified",
        "employment_type": "Not Specified",
        "apply_link": link
    }
    
    try:
        await page.goto(link, timeout=15000)
        
        # --- WUZZUF EXTRACTION ---
        if source == "Wuzzuf":
            # 1. Capture Full Description (Unchanged)
            try:
                elm = await page.wait_for_selector(".css-1uobp1k", timeout=4000)
                data["description"] = await elm.inner_text()
            except: 
                data["description"] = "Description not found."
            
            # 2. Extract New Data (Skills)
            try:
                skill_tags = await page.query_selector_all(".css-158idm4") 
                if not skill_tags: skill_tags = await page.query_selector_all("a.css-o171kl")
                data["skills"] = [(await t.inner_text()).strip() for t in skill_tags]
            except: pass

            # 3. Extract New Data (Experience/Type)
            try:
                career_el = await page.query_selector("span:has-text('Career Level:') + span")
                if career_el: data["experience_level"] = await career_el.inner_text()
                
                type_el = await page.query_selector("span:has-text('Job Type:') + span")
                if type_el: data["employment_type"] = await type_el.inner_text()
            except: pass

        # --- LINKEDIN EXTRACTION ---
        elif source == "LinkedIn":
            # 1. Capture Full Description (Unchanged)
            try:
                try: await page.click("button.show-more-less-html__button", timeout=1500)
                except: pass
                elm = await page.wait_for_selector(".show-more-less-html__markup, .description__text", timeout=4000)
                data["description"] = await elm.inner_text()
            except: 
                 data["description"] = "Description not found."
            
            # 2. Extract Apply Link
            try:
                apply_btn = await page.query_selector("a.top-card-layout__cta--primary, a.apply-button")
                if apply_btn:
                    href = await apply_btn.get_attribute("href")
                    if href and "linkedin.com" not in href: data["apply_link"] = href
            except: pass
            
            # 3. Extract New Data (Criteria List)
            try:
                criteria_list = await page.query_selector_all(".description__job-criteria-item")
                for item in criteria_list:
                    header = await item.query_selector("h3")
                    val = await item.query_selector("span")
                    if header and val:
                        h_text = (await header.inner_text()).lower()
                        v_text = (await val.inner_text()).strip()
                        if "seniority" in h_text: data["experience_level"] = v_text
                        if "employment" in h_text: data["employment_type"] = v_text
            except: pass

        # --- FINAL PARSING ---
        # Parse requirements/responsibilities from the full description
        # But KEEP data['description'] intact.
        if data["description"]:
            reqs, resps = parse_description(data["description"])
            data["requirements"] = reqs if reqs else "See Description"
            data["responsibilities"] = resps if resps else "See Description"

        await page.close()
        return data
        
    except Exception as e:
        await page.close()
        return data

# --- ASYNC SCRAPER LOGIC ---
async def run_scraper():
    joblens = JobLensScraper()
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36")
        page = await context.new_page()
        await stealth_async(page)

        # 1. WUZZUF
        print("\n--- 1. Scraping Wuzzuf ---")
        try:
            await page.goto("https://wuzzuf.net/search/jobs/?q=&a=hpb&filters%5Bcountry%5D%5B0%5D=Egypt", timeout=60000)
            try: await page.wait_for_selector(".css-1gatmva, div.css-pkv5jc", timeout=20000)
            except: print("   ⚠️ Wuzzuf selector timeout.")

            cards = await page.query_selector_all(".css-1gatmva")
            if not cards: cards = await page.query_selector_all("div.css-pkv5jc")
            
            print(f"   Found {len(cards)} job cards. Fetching details...")
            jobs = []
            for i, card in enumerate(cards):
                t_el = await card.query_selector("h2")
                c_el = await card.query_selector(".css-17s97q8")
                l_el = await card.query_selector("h2 a")
                
                title = await t_el.inner_text() if t_el else "N/A"
                company = await c_el.inner_text() if c_el else "N/A"
                post_link = await l_el.get_attribute("href") if l_el else ""
                
                print(f"   [{i+1}/{len(cards)}] Fetching: {title[:30]}...", end="\r")
                
                # Get Details (Description + New Structured Data)
                details = await get_job_details(context, post_link, "Wuzzuf")

                jobs.append({
                    "source": "Wuzzuf", 
                    "title": title.strip(), 
                    "company": company.replace("-", "").strip(), 
                    "location": "Egypt", 
                    "job_page_link": post_link, 
                    "posted_time": "Recent",
                    **details # Unpack details (description, skills, experience, etc.)
                })
            
            print(f"\n   ✅ Extracted {len(jobs)} Wuzzuf jobs.")
            joblens.save_jobs(jobs)

        except Exception as e: print(f"❌ Wuzzuf Error: {e}")

        # 2. LINKEDIN
        print("\n--- 2. Scraping LinkedIn ---")
        try:
            await page.goto("https://www.linkedin.com/jobs/search?keywords=&location=Egypt&position=1&pageNum=0", timeout=60000)
            for _ in range(5):
                await page.mouse.wheel(0, 1000)
                await asyncio.sleep(2)

            cards = await page.query_selector_all(".base-card")
            print(f"   Found {len(cards)} job cards. Fetching details...")

            jobs = []
            for i, card in enumerate(cards):
                try:
                    t_el = await card.query_selector(".base-search-card__title")
                    c_el = await card.query_selector(".base-search-card__subtitle")
                    l_el = await card.query_selector("a.base-card__full-link")
                    
                    if t_el and c_el and l_el:
                        post_link = await l_el.get_attribute("href")
                        title = (await t_el.inner_text()).strip()
                        
                        print(f"   [{i+1}/{len(cards)}] Fetching: {title[:30]}...", end="\r")
                        
                        details = await get_job_details(context, post_link, "LinkedIn")

                        jobs.append({
                            "source": "LinkedIn",
                            "title": title,
                            "company": (await c_el.inner_text()).strip(),
                            "location": "Egypt",
                            "job_page_link": post_link,
                            "posted_time": "Recent",
                            **details
                        })
                except: continue
            
            print(f"\n   ✅ Extracted {len(jobs)} LinkedIn jobs.")
            joblens.save_jobs(jobs)
        except Exception as e: print(f"❌ LinkedIn Error: {e}")

        await browser.close()

d:\anaconda\envs\joblens\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Cell 3: The Execution Wrapper
import sys
from threading import Thread

def run_scraper_thread(coro):
    """
    Runs an async function in a separate thread to avoid the 
    Windows NotImplementedError in Jupyter.
    """
    def target():
        # Force the correct Windows policy inside this thread
        if sys.platform == 'win32':
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        
        # Create a fresh loop
        new_loop = asyncio.new_event_loop()
        asyncio.set_event_loop(new_loop)
        
        # Run the scraper
        try:
            new_loop.run_until_complete(coro)
        finally:
            new_loop.close()

    t = Thread(target=target)
    t.start()
    t.join()

print("🚀 Starting scraper...")

# THIS REPLACES 'await run_scraper()'
run_scraper_thread(run_scraper())

print("✅ Finished.")

🚀 Starting scraper...
Loading AI Model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 493.89it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 97 existing jobs.

--- 1. Scraping Wuzzuf ---
   Found 15 job cards. Fetching details...
   [15/15] Fetching: senior marketing...ator...- Di...
   ✅ Extracted 15 Wuzzuf jobs.
   Processing 15 candidates for database...
   ✅ Success: Added 15 NEW jobs to DB.

--- 2. Scraping LinkedIn ---
   Found 60 job cards. Fetching details...
   [60/60] Fetching: Site Civil Engineer...ly...Mer...
   ✅ Extracted 60 LinkedIn jobs.
   Processing 60 candidates for database...
   ✅ Success: Added 2 NEW jobs to DB.
✅ Finished.


In [4]:
import chromadb
import json

# 1. Connect to the SAME database path used in your scraper
CHROMA_PATH = "./joblens_db"
client = chromadb.PersistentClient(path=CHROMA_PATH)

# 2. Get the collection
try:
    collection = client.get_collection(name="job_listings")
    
    # 3. Check the Count
    count = collection.count()
    print(f"✅ CONNECTION SUCCESSFUL")
    print(f"📊 Total Jobs in Database: {count}")

    if count > 0:
        # 4. Peek at the Data (Show the first 3 items)
        print("\n🔍 Inspecting first 3 jobs:")
        data = collection.peek(limit=3)
        
        # 'data' is a dictionary of lists. We iterate through them.
        for i in range(len(data['ids'])):
            print("-" * 40)
            print(f"ID:     {data['ids'][i]}")
            print(f"Source: {data['metadatas'][i]['source']}")
            print(f"Title:  {data['metadatas'][i]['title']}")
            print(f"Link:   {data['metadatas'][i]['link']}")
            
            # Optional: Verify the JSON detail is valid
            try:
                details = json.loads(data['metadatas'][i]['json_detailed'])
                print(f"JSON Check: Valid ✅ (Location: {details.get('location', 'N/A')})")
            except:
                print("JSON Check: Invalid ❌")
    else:
        print("\n⚠️ The database is empty! Run 'smart_scraper.py' first.")

except Exception as e:
    print(f"❌ Error: Could not load collection. Reason: {e}")

✅ CONNECTION SUCCESSFUL
📊 Total Jobs in Database: 114

🔍 Inspecting first 3 jobs:
----------------------------------------
ID:     a61c0b6aca4b5c61d9dab81bf76351e2
Source: Wuzzuf
Title:  Customer Support Agent (Chat Account \ B1+ English)
Link:   https://wuzzuf.net/jobs/p/l22ctxuywoz7-customer-support-agent-chat-account-b1-english-etisalat-egypt-cairo-egypt
JSON Check: Valid ✅ (Location: Egypt)
----------------------------------------
ID:     8774fce22ad85eb5763d0db5e608ebc6
Source: Wuzzuf
Title:  Chat Support Representative (B1+ English Level)
Link:   https://wuzzuf.net/jobs/p/ip6cscitop1a-chat-support-representative-b1-english-level-etisalat-egypt-cairo-egypt
JSON Check: Valid ✅ (Location: Egypt)
----------------------------------------
ID:     7a1d7065f0b6dc97305b1e095f41beec
Source: Wuzzuf
Title:  OPD Coordinator
Link:   https://wuzzuf.net/jobs/p/bm0fbtqdyshg-opd-coordinator-saudi-german-hospital-cairo-egypt
JSON Check: Valid ✅ (Location: Egypt)


In [1]:
# Cell 5: JobLens API (Complete: Search, Scraper, Create, Update, Delete)
import uvicorn
import uuid
import datetime
from fastapi import FastAPI, HTTPException, BackgroundTasks, Query
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import chromadb
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
CHROMA_PATH = "./joblens_db"
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# --- APP SETUP ---
app = FastAPI(title="JobLens API")

print("⏳ Loading AI Model & Database...")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
# We use "job_listings" to match your scraper's collection
collection = chroma_client.get_or_create_collection(name="job_listings")
model = SentenceTransformer(EMBEDDING_MODEL)
print("✅ System Ready.")

# --- MODELS ---
class JobData(BaseModel):
    title: str
    description: str
    requirements: Optional[str] = "Not Specified"
    responsibilities: Optional[str] = "Not Specified"
    skills: List[str] = []
    location: str
    experience_level: str = "Not Specified"
    employment_type: str = "Not Specified"
    company_name: str

class JobEmbeddingRequest(BaseModel):
    job_id: int
    job_data: JobData

class ScrapingTriggerRequest(BaseModel):
    sources: List[str] = ["linkedin", "wuzzuf"]
    keywords: List[str]
    locations: List[str] = ["Egypt"]
    max_pages: int = 1

# --- HELPER: SHARED LOGIC ---
def process_and_store_job(job_id: int, job_data: JobData):
    """Reusable logic for both Create and Update endpoints"""
    # 1. Create rich text for the AI
    skills_str = ", ".join(job_data.skills)
    full_text = (
        f"{job_data.title} at {job_data.company_name}. "
        f"Location: {job_data.location}. "
        f"Level: {job_data.experience_level}. "
        f"Skills: {skills_str}. "
        f"Description: {job_data.description} "
        f"Requirements: {job_data.requirements}"
    )

    # 2. Generate Embedding
    embedding = model.encode(full_text).tolist()
    
    # 3. Store (Upsert)
    # We prefix with "job_" to ensure ID is a string (Chroma requirement)
    embedding_id = f"job_{job_id}"
    
    metadata = {
        "original_job_id": job_id,
        "title": job_data.title,
        "company": job_data.company_name,
        "location": job_data.location,
        "experience_level": job_data.experience_level,
        "source": "Internal API", # Mark these as manually added
        "skills_list": skills_str
    }

    collection.upsert(
        ids=[embedding_id],
        embeddings=[embedding],
        documents=[full_text],
        metadatas=[metadata]
    )
    return embedding_id

# --- HELPER: BACKGROUND TASK ---
async def run_background_scraper(params: ScrapingTriggerRequest):
    print(f"🔄 BACKGROUND TASK: Scraping {params.keywords}...")
    # This calls your scraping logic. In a real app, you'd pass params to run_scraper()
    await run_scraper() 
    print("✅ BACKGROUND TASK FINISHED.")

# --- ENDPOINTS ---

# 1. GET SCRAPED JOBS (Search)
@app.get("/api/scraping/jobs")
async def get_scraped_jobs(
    keyword: Optional[str] = Query(None),
    location: Optional[str] = Query(None),
    limit: int = 50
):
    try:
        where_filter = {}
        if location: where_filter["location"] = location
        if len(where_filter) == 0: where_filter = None

        if keyword:
            # Semantic Search
            query_vec = model.encode(keyword).tolist()
            results = collection.query(
                query_embeddings=[query_vec], n_results=limit, where=where_filter
            )
            ids = results['ids'][0]
            metas = results['metadatas'][0]
        else:
            # List All
            results = collection.get(limit=limit, where=where_filter)
            ids = results['ids']
            metas = results['metadatas']

        formatted_jobs = []
        for i in range(len(ids)):
            m = metas[i]
            formatted_jobs.append({
                "external_job_id": ids[i],
                "title": m.get("title", "N/A"),
                "company": m.get("company", "N/A"),
                "location": m.get("location", "N/A"),
                "description": m.get("description_snippet", "")[:100] + "...",
                "skills": m.get("skills_list", "").split(", ")
            })
            
        return {"success": True, "data": formatted_jobs, "count": len(ids)}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 2. TRIGGER SCRAPING
@app.post("/api/scraping/trigger", status_code=202)
async def trigger_scraping_job(payload: ScrapingTriggerRequest, bt: BackgroundTasks):
    bt.add_task(run_background_scraper, payload)
    return {"success": True, "message": "Scraping queued"}

# 3. CREATE EMBEDDING
@app.post("/api/embeddings/job")
async def create_job_embedding(payload: JobEmbeddingRequest):
    try:
        eid = process_and_store_job(payload.job_id, payload.job_data)
        return {"success": True, "message": "Created", "data": {"id": eid}}
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

# 4. UPDATE EMBEDDING (RESTORED)
@app.put("/api/embeddings/job/{job_id}")
async def update_job_embedding(job_id: int, payload: JobEmbeddingRequest):
    try:
        if job_id != payload.job_id:
            raise HTTPException(status_code=400, detail="URL ID mismatch")
        
        process_and_store_job(payload.job_id, payload.job_data)
        return {"success": True, "message": "Job embedding updated successfully"}
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

# 5. DELETE EMBEDDING
@app.delete("/api/embeddings/job/{job_id}")
async def delete_job_embedding(job_id: int):
    try:
        collection.delete(ids=[f"job_{job_id}"])
        return {"success": True, "message": "Deleted"}
    except Exception as e: raise HTTPException(status_code=500, detail=str(e))

# --- EXECUTION ---
print("🚀 JobLens API running at http://127.0.0.1:8000")
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()

d:\anaconda\envs\joblens\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading AI Model & Database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 995.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ System Ready.
🚀 JobLens API running at http://127.0.0.1:8000
